# libpostal vs Step 1b — side by side
Compares the Step 2c libpostal rule with the Step 1b similarity models (LR / RF / XGB) on the same val and test splits, for both datasets. Loads predictions and features only — libpostal predictions come from the cache built by `dumb_libpostal_baseline.ipynb`; the Step 1b models are fit here once. The reported numbers live in **`results.md`** (project root); this notebook is where they come from. Explorations: **Diagnostics** (last cell).

## 0. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import pandas as pd
pd.set_option("display.width", 250)              # widen the line
pd.set_option("display.expand_frame_repr", False)  # never wrap, scroll sideways

from helpers.pairs import load_pairs, add_number_street
from helpers.features import get_processed_dfs
from helpers.check_model_performance import review_typical_slices
from process_dumb_models import run_against_models
from model_baseline.dumb_libpostal_model import _get_libpostal_results

## 1. Load data + features

In [ ]:
ot_train, ot_train_label = load_pairs("overture", "train")
ot_val, ot_val_label = load_pairs("overture", "val")
ot_test, ot_test_label = load_pairs("overture", "test")

st_train, st_train_label = load_pairs("stamford", "train")
st_val, st_val_label = load_pairs("stamford", "val")
st_test, st_test_label = load_pairs("stamford", "test")

# House number + street remainder for the similarity features (helpers/pairs.py):
# adds number_a/b, street_a/b, number_missing, both_number_missing
ot_train, ot_val, ot_test = [add_number_street(df) for df in [ot_train, ot_val, ot_test]]
st_train, st_val, st_test = [add_number_street(df) for df in [st_train, st_val, st_test]]

In [ ]:
# Similarity features (cached; ~1.5 min for Overture on first run)
train_overture, val_overture, test_overture = get_processed_dfs([
    [ot_train, "./overture_training/overture_evaluation_train.parquet"],
    [ot_val, "./overture_training/overture_evaluation_val.parquet"],
    [ot_test, "./overture_training/overture_evaluation_test.parquet"],
])
train_stamford, val_stamford, test_stamford = get_processed_dfs([
    [st_train, "./stamford_training/evaluation_train.parquet"],
    [st_val, "./stamford_training/evaluation_val.parquet"],
    [st_test, "./stamford_training/evaluation_test.parquet"],
])

## 2. Step 1b models

In [ ]:
# Step 1b models (LR / RF / XGB): fit on train once, predict val and test (this is Step 1b's single test run)
def _deconstruct_dumb_model_results(ret):
    # {model: {phase: [preds, probs]}} -> {"model|phase": preds}
    all_preds = {}
    for model, phases in ret.items():
        for phase, (preds, _probs) in phases.items():
            all_preds[f"{model}|{phase}"] = preds
    return all_preds

ret_dumb_overture = _deconstruct_dumb_model_results(run_against_models(train_overture, ot_train_label, val_overture, test_overture))
ret_dumb_stamford = _deconstruct_dumb_model_results(run_against_models(train_stamford, st_train_label, val_stamford, test_stamford))

## 3. libpostal vs LR, both datasets

In [ ]:
# Which libpostal setting (road, zip, number) and which Step 1b model to put side by side
FIXED_LIBPOSTAL_OPTIONS = ["loose", "strict", "strict"]
FIXED_DUMB_BASELINE_MODEL = "LogisticRegression"
LIB, BASE = "lib", "LR"

def _pct_col(df, preds, labels, name):
    # "% correct" column of review_typical_slices as floats (99.0, not "99.0%"), indexed by slice, labelled `name`
    pct = review_typical_slices(df, preds, labels).set_index("slice")["% correct"]
    return pct.str[:-1].astype(float).rename(name)

def compare_lib_vs_baseline(val, val_label, test, test_label, ret_dumb, dataset):
    road, zip_rule, number = FIXED_LIBPOSTAL_OPTIONS
    lib_val = _get_libpostal_results(val, road, zip_rule, number, "val", dataset=dataset)
    lib_test = _get_libpostal_results(test, road, zip_rule, number, "test", dataset=dataset)
    table = pd.concat([
        _pct_col(val, lib_val, val_label, f"{LIB}(val)"),
        _pct_col(test, lib_test, test_label, f"{LIB}(test)"),
        _pct_col(val, ret_dumb[f"{FIXED_DUMB_BASELINE_MODEL}|val"], val_label, f"{BASE}(val)"),
        _pct_col(test, ret_dumb[f"{FIXED_DUMB_BASELINE_MODEL}|test"], test_label, f"{BASE}(test)"),
    ], axis=1)
    # Plain mean of val and test (unweighted by n)
    table[f"{LIB} avg"] = table[[f"{LIB}(val)", f"{LIB}(test)"]].mean(axis=1)
    table[f"{BASE} avg"] = table[[f"{BASE}(val)", f"{BASE}(test)"]].mean(axis=1)
    # |val - test|: how much each model's score moves between sets
    table[f"{LIB} val/test gap"] = (table[f"{LIB}(val)"] - table[f"{LIB}(test)"]).abs()
    table[f"{BASE} val/test gap"] = (table[f"{BASE}(val)"] - table[f"{BASE}(test)"]).abs()
    # Signed: positive = libpostal better than the baseline, negative = baseline better
    table[f"{BASE}->{LIB} diff"] = table[f"{LIB} avg"] - table[f"{BASE} avg"]
    return table, lib_val["pred"], lib_test["pred"]

compare_overture, ot_lib_val, ot_lib_test = compare_lib_vs_baseline(ot_val, ot_val_label, ot_test, ot_test_label, ret_dumb_overture, "Overture")
compare_stamford, st_lib_val, st_lib_test = compare_lib_vs_baseline(st_val, st_val_label, st_test, st_test_label, ret_dumb_stamford, "Stamford")

# Floats stay in the tables; format as "xx.x%" only for display
for name, table in [("Overture", compare_overture), ("Stamford", compare_stamford)]:
    print(f"{'='*20} {name} {'='*20}")
    print(table.apply(lambda col: col.map("{:.1f}%".format)).to_string())

## Diagnostics
One-off checks from the comparison (2026-10-15), each as **question → result → query**. LR = Step 1b logistic regression; lib = libpostal at the `FIXED_LIBPOSTAL_OPTIONS` setting (loose/strict/strict). Masks below use `ret_dumb_*` and `*_lib_*` from the cells above.

**1. Why does LR find ~20 pts more non-identical matches than libpostal (Overture)?** The pairs LR gets right and libpostal gets wrong are small differences in the street (typos, suffix forms the parser keeps as different names). libpostal's road check is categorical: a typo makes the road NON and the pair is rejected outright, with no partial credit. LR scores *how similar the letters are*, so `CONNECTICUT` / `CONNETICUT` (~0.97) still passes.
```python
lr = ret_dumb_overture["LogisticRegression|val"]
mask = ((lr == ot_val_label) & (ot_lib_val != ot_val_label) & (ot_val["address_a"] != ot_val["address_b"])).values
ot_val.assign(label=ot_val_label, lr_guess=lr, lib_guess=ot_lib_val)[mask][["address_a", "address_b", "lr_guess", "lib_guess", "label"]]
```

**2. Why does libpostal reject ~60 pts more case2 than LR (Overture)?** Not abbreviation knowledge. On e.g. `70 MAIN ST` / `70 PINE ST` and `38 ORANGE AVE` / `38 OCEAN AVE`, libpostal parses the road and asks whether it is the *same name* — `main st` vs `pine st` → NON → reject. LR sees house number identical (1.0), ZIP identical (1.0) and look-alike letters (street JW ≈ 0.8–0.9) → match. Case2 is built from exactly such pairs (same number + ZIP, street JW > 0.8), so the property that wins non-identical positives for LR (graded letter similarity) is the one that loses case2. Val: 16 case2 rows LR wrong / libpostal right — mostly mirrored pairs (`220 CHAPEL RD` / `220 HALE RD` both ways, `191 MAIN` / `191 DEMING`, `35 BUNKER HILL` / `35 CANDEE HILL`). The test-mask bug that printed an empty frame (`&` before `==`) is fixed in the query.
```python
lr = ret_dumb_overture["LogisticRegression|val"]
mask = ((lr != ot_val_label) & (ot_lib_val == ot_val_label) & (ot_val["neg_type"] == "case2")).values
ot_val.assign(label=ot_val_label, lr_guess=lr, lib_guess=ot_lib_val)[mask][["address_a", "address_b", "zip_a", "zip_b", "lr_guess", "lib_guess", "label"]]
```

**3. Are Overture and Stamford case2 equally hard?** Not at first. Median street Jaro-Winkler (strings the model sees), val — Overture 0.88 (n = 34); Stamford 0.74 (n = 190) with the first derived rule (same number, *any* different street). After tightening Stamford case2 to Overture's rule (clean-street JW > 0.8, `Stamford/03_convert_to_shared_schema.py`): Stamford 0.81 (n = 96) *(Claude)* — closer, still lower (side B is corrupted). Train case2: Overture 172 of 202,568 rows; Stamford 1,779 of 15,225 after tightening (3,537 before).
```python
import jellyfish
for name, df in [("overture val", ot_val), ("stamford val", st_val)]:
    c2 = df[df["neg_type"] == "case2"]
    jw = [jellyfish.jaro_winkler_similarity(a, b) for a, b in zip(c2["street_a"], c2["street_b"])]
    print(name, len(c2), pd.Series(jw).describe()[["25%", "50%", "75%"]].round(2).to_dict())
print((ot_train["neg_type"] == "case2").sum(), (st_train["neg_type"] == "case2").sum())
```

**4. Why does LR reject case2 on Stamford (~93%) but not on Overture (~5%)?** David's conclusion: training-data imbalance is the leading cause — LR sees ~1,800 case2 pairs in Stamford train vs 172 in Overture train, and to LR `PINE` vs `MAIN` is just a similarity number, so it needs examples to learn where the line is. Not isolated from side-B corruption lowering similarity (3). Controlled test deferred: up-weight case2 in Overture train and compare case2 rejection / non-identical recall on val.

**5. What do the Stamford pairs LR gets right and libpostal wrong look like?** val 50 rows, test 32 — all true matches with random character edits in the street (`ALMA ROCK` / `AXMA ROCK ROAD`, `ATLANTIC` / `ATLAXTIC`, `WESTOVER` / `WESTOVR`): libpostal's name check rejects any edited word; LR's letter similarity tolerates it. This is why libpostal finds 39% of Stamford non-identical matches vs 52–72% on Overture, where differences are mostly formatting (`ST` / `STREET`, units) that libpostal normalises.
```python
lr = ret_dumb_stamford["LogisticRegression|val"]
mask = ((lr == st_val_label) & (st_lib_val != st_val_label)).values
st_val.assign(label=st_val_label, lr_guess=lr, lib_guess=st_lib_val)[mask][["address_a", "address_b", "zip_a", "zip_b", "lr_guess", "lib_guess", "label"]].sample(frac=1, random_state=0).head(10)
```

**Removed from this notebook:** the 8-setting libpostal val-vs-test grid (identical to §3 of `dumb_libpostal_baseline.ipynb`).